# Spark BigQuery Connector 실습 : Write (Read 포함)
Spark Batch Write 실습 (Indirect / Direct / Direct vs Indirect)

**목표** : 같은 DataFrame 을 `writeMethod` 만 바꿔서 두 경로로 쓰고, 차이를 눈으로 확인한다.

| 구분 | Indirect | Direct |
|---|---|---|
| 경로 | GCS Staging → BigQuery Load Job | Storage Write API |
| 중간 저장소 | GCS 필요 | 없음 |
| 추가 설정 | `temporaryGcsBucket` (또는 `persistentGcsBucket`) 필요 | 불필요 |

**실습 순서**
1. 설정 / Spark 세션
2. **Read** : 기본 읽기 → 옵션으로 필터 지정 / SQL 결과 읽기
3. **Write** : ① 설정 차이 → ② GCS 흔적 → ③ BigQuery Job 기록 → ④ 적재 결과 확인 → Scorecard
4. 보충(선택) : `mode` 비교, 파티션 / 클러스터링 옵션

**사전 준비 체크리스트**
- 리전은 **asia-northeast3 (서울)** 로 통일 : Spark 세션 / GCS 버킷 / BigQuery 데이터셋
- 노트북 실행 계정(서비스 계정)에 아래 권한
  - BigQuery Data Editor, BigQuery Job User, BigQuery Read Session User
  - 버킷에 Storage Object Admin
  - (③ Job 기록 조회) `bigquery.jobs.listAll` 이 포함된 권한 (예: BigQuery Resource Viewer)
- Spark 세션은 시작에 시간이 걸리므로 **강의 전에 미리 연결**해 둔다.

## 0. 설정
`YOUR_PROJECT_ID`, `YOUR_BUCKET` 을 환경에 맞게 바꾼다.
(`thelook_ecommerce` 데이터셋은 앞선 External Table / BigLake 실습에서 복사해 둔 테이블을 그대로 사용한다.)

In [ ]:
import time
from google.cloud import bigquery

PROJECT  = "YOUR_PROJECT_ID"
BUCKET   = "YOUR_BUCKET"          # asia-northeast3 버킷
LOCATION = "asia-northeast3"
DATASET  = "spark_demo"           # 이번 실습에서 새로 만드는 데이터셋
WRITE_SRC = f"{PROJECT}.thelook_ecommerce.orders"   # Write 에 사용할 원본 테이블 (큰 데이터 비교는 events 로 변경)

RUN_ID = time.strftime("%Y%m%d_%H%M%S")

bq = bigquery.Client(project=PROJECT, location=LOCATION)

ds = bigquery.Dataset(f"{PROJECT}.{DATASET}")
ds.location = LOCATION
bq.create_dataset(ds, exists_ok=True)
print(f"dataset ready: {PROJECT}.{DATASET} ({LOCATION})")

## 1. Spark 세션 확인
BigQuery Studio 노트북에서 Serverless Spark 세션을 연결하면 `spark` 변수가 이미 있다.
(없을 때만 아래 셀이 세션을 만든다. 이 경우 `pip install dataproc-spark-connect` 가 필요할 수 있다.)

In [ ]:
try:
    spark
except NameError:
    from google.cloud.dataproc_spark_connect import DataprocSparkSession
    spark = DataprocSparkSession.builder.getOrCreate()

print("Spark version:", spark.version)

## 2. Read : BigQuery → Spark DataFrame
Spark BigQuery Connector 는 `format("bigquery")` 로 읽고 쓴다.
읽기는 BigQuery Storage Read API 를 사용하며, **필요한 컬럼과 필터만 BigQuery 쪽에서 걸러서** 가져온다.

### 2-1. 테이블 기본 읽기

In [ ]:
SRC_ORDERS = f"{PROJECT}.thelook_ecommerce.orders"

df_orders = (
    spark.read.format("bigquery")
    .option("table", SRC_ORDERS)
    .load()
)

df_orders.printSchema()
print("rows:", df_orders.count())
df_orders.show(5, truncate=False)

### 2-2. 옵션으로 필터 지정 / SQL 결과 읽기
- **(a) `filter` 옵션** : 조건이 BigQuery 로 내려가서(Pushdown) 읽는 데이터 양 자체가 줄어든다. `select` 로 필요한 컬럼만 가져오면 더 줄어든다.
- **(b) `query` 옵션** : 테이블이 아니라 **SQL 결과**를 읽는다. 결과를 임시 테이블로 만들 데이터셋(`materializationDataset`)이 필요하고, 같은 리전이어야 한다.
  (임시 테이블은 기본 설정으로 일정 시간 뒤 자동 만료된다.)

In [ ]:
# (a) 옵션으로 필터 지정
df_complete = (
    spark.read.format("bigquery")
    .option("table", SRC_ORDERS)
    .option("filter", "status = 'Complete' AND created_at >= '2024-01-01'")
    .load()
    .select("order_id", "user_id", "status", "created_at")
)
print("Complete & 2024+ :", df_complete.count())

# (b) SQL로 조회
df_status = (
    spark.read.format("bigquery")
    .option("viewsEnabled", "true")
    .option("materializationDataset", DATASET)  # 결과를 임시테이블로 만들고 그 결과를 출력 하기 위한 dataset
    .option("query", f"""
        SELECT status, COUNT(*) AS order_count
        FROM `{SRC_ORDERS}`
        GROUP BY status
        ORDER BY order_count DESC
    """)
    .load()
)

df_status.show()

## 3. Write 비교 준비
- **적재 대상** : 원본 테이블을 **변환 없이 그대로** 읽어서 `spark_demo` 에 적재한다.
  `cache()` 후 `count()` 로 미리 읽어 두어, 두 Write 가 같은 원본을 같은 방식으로 쓰게 한다.
- 이번 실습의 대상 테이블(`orders_direct`, `orders_indirect`)만 재실행 때 초기화한다. (다른 기존 객체는 건드리지 않는다.)
- 테이블 초기화 함수만 만든다. 결과 확인(GCS / Job / 행 수)은 콘솔과 SQL 로 한다.

In [ ]:
df_write = (
    spark.read.format("bigquery")
    .option("table", WRITE_SRC)
    .load()
    .cache()
)

WRITE_ROWS = df_write.count()
print("적재 대상 행 수:", WRITE_ROWS)
df_write.printSchema()

In [ ]:
def drop_tables(*names):
    for n in names:
        bq.delete_table(f"{PROJECT}.{DATASET}.{n}", not_found_ok=True)

drop_tables("orders_direct", "orders_indirect")
print("ready")

## 4. Write 비교

### 4-① 설정 차이 : Direct 는 버킷 없이 바로 쓴다 ("추가 설정")
**Direct Write** : `writeMethod = direct` → GCS 도 필요 없고, Storage Write API 로 바로 적재한다.

In [ ]:
(
    df_write.write.format("bigquery")
    .option("writeMethod", "direct")
    .option("table", f"{PROJECT}.{DATASET}.orders_direct")
    .mode("append")
    .save()
)

print("Direct 완료 : 중간 저장소 설정 없음")

**Indirect Write 를 버킷 설정 없이 실행하면 에러가 나는 것이 정상이다.**
→ Indirect 는 GCS Staging 이 필수이기 때문.

In [ ]:
try:
    (
        df_write.write.format("bigquery")
        .option("writeMethod", "indirect")
        .option("table", f"{PROJECT}.{DATASET}.orders_indirect")
        .mode("append")
        .save()
    )
    print("⚠ 에러가 나지 않았다. 세션 기본 설정에 버킷이 지정되어 있는지 확인")
except Exception as e:
    print("예상된 에러 (GCS 버킷 미지정):")
    print(str(e)[:600])

# 혹시 일부라도 만들어졌다면 정리하고 다음 단계로
drop_tables("orders_indirect")

### 4-② Indirect Write : GCS Staging + Load Job
강의 자료의 예제는 `temporaryGcsBucket` 을 쓴다. 이 옵션은 적재 후 **임시 파일을 자동 삭제**하므로 실습에서는 흔적을 볼 수 없다.
그래서 여기서는 같은 동작을 하되 파일을 남기는 `persistentGcsBucket` + `persistentGcsPath` 를 쓴다.
(실무에서는 슬라이드처럼 `temporaryGcsBucket` 을 쓰면 된다.)

In [ ]:
STAGE_PATH = f"spark_demo/indirect_stage_{RUN_ID}"

(
    df_write.write.format("bigquery")
    .option("writeMethod", "indirect")
    .option("persistentGcsBucket", BUCKET)          # 슬라이드: .option("temporaryGcsBucket", "my-temp-bucket")
    .option("persistentGcsPath", STAGE_PATH)
    .option("table", f"{PROJECT}.{DATASET}.orders_indirect")
    .mode("append")
    .save()
)

print(f"gs://{BUCKET}/{STAGE_PATH}/")   # 이 경로를 콘솔에서 확인

## 5. 눈으로 확인하기

### ② GCS 흔적 (콘솔)
1. 콘솔 → **Cloud Storage → 버킷** (`YOUR_BUCKET`) 으로 이동
2. 위 셀에서 출력된 경로 (`spark_demo/indirect_stage_<RUN_ID>/`) 로 들어간다
3. Parquet 파일이 생긴 것을 확인한다 → **Indirect 는 GCS 를 거쳤다**
4. Direct 는 이 경로가 없다 (GCS 를 쓰지 않음)

### ③ BigQuery Job 기록 (SQL)
콘솔 **BigQuery → 쿼리 작성** 에 아래 쿼리를 붙여넣어 실행한다.
Indirect 는 **LOAD Job** 이 보여야 하고, Direct 는 LOAD Job 이 보이지 않아야 한다.
(Storage Write API 의 Job 표시 방식은 문서 기준이며 **리허설에서 확인 필요**)

```sql
-- 최근 3시간 동안 spark_demo 의 두 테이블에 대해 실행된 Job
SELECT
    creation_time,
    job_id,
    job_type,
    state,
    destination_table.table_id AS table_id
FROM `region-asia-northeast3`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 3 HOUR)
  AND destination_table.dataset_id = 'spark_demo'
  AND destination_table.table_id IN ('orders_direct', 'orders_indirect')
ORDER BY creation_time;
```

원본과 두 테이블의 행 수가 모두 같아야 한다.

```sql
SELECT 'source (thelook orders)' AS table_name, COUNT(*) AS row_cnt
FROM `YOUR_PROJECT_ID.thelook_ecommerce.orders`
UNION ALL
SELECT 'orders_direct', COUNT(*)
FROM `YOUR_PROJECT_ID.spark_demo.orders_direct`
UNION ALL
SELECT 'orders_indirect', COUNT(*)
FROM `YOUR_PROJECT_ID.spark_demo.orders_indirect`;
```

| 항목 | Indirect | Direct |
|---|---|---|
| `writeMethod` | `indirect` | `direct` |
| 중간 저장소 (GCS) | 필요 (`persistentGcsBucket`) | 없음 |
| GCS Staging 파일 | 생성됨 (② 콘솔) | 없음 |
| BigQuery LOAD Job | 있음 (③ SQL) | 없음 (문서 기준, 리허설에서 확인 필요) |
| 적재 행 수 | 원본과 동일 (④ SQL) | 원본과 동일 (④ SQL) |

> 위 표는 ②~④ 에서 콘솔과 SQL 로 확인한 결과를 대조하는 용도다.

## 7. 보충 (선택)

### 7-1. `mode` 비교 : append / overwrite
같은 데이터를 `append` 로 두 번 쓰면 **중복**되고, `overwrite` 는 **교체**한다.
(여기서는 슬라이드와 같은 `temporaryGcsBucket` 방식의 Indirect 를 사용한다.)

In [ ]:
MODE_TABLE = f"{PROJECT}.{DATASET}.orders_mode_demo"
drop_tables("orders_mode_demo")

df_small = df_write.limit(1000)

for mode in ["append", "append", "overwrite"]:
    (
        df_small.write.format("bigquery")
        .option("writeMethod", "indirect")
        .option("temporaryGcsBucket", BUCKET)
        .option("table", MODE_TABLE)
        .mode(mode)
        .save()
    )

SQL 로 행 수 확인 (append → append → overwrite 순서로 실행한 뒤)

```sql
SELECT COUNT(*) AS row_cnt
FROM `YOUR_PROJECT_ID.spark_demo.orders_mode_demo`;
```

예상 결과 : `1000` (마지막 `overwrite` 가 테이블을 교체). 중복(`2000`)을 보려면 마지막 `overwrite` 줄을 빼고 실행한 뒤 같은 쿼리를 실행한다.

### 7-2. 파티션 / 클러스터링 옵션으로 적재
앞의 Partitioning / Clustering 내용과 연결한다.
Connector 옵션 `partitionField`, `partitionType`, `clusteredFields` 로 **적재와 동시에** 파티션 테이블을 만든다.

In [ ]:
PART_TABLE = f"{PROJECT}.{DATASET}.orders_partitioned_spark"
drop_tables("orders_partitioned_spark")

(
    df_write.write.format("bigquery")
    .option("writeMethod", "indirect")
    .option("temporaryGcsBucket", BUCKET)
    .option("table", PART_TABLE)
    .option("partitionField", "created_at")      # TIMESTAMP 컬럼
    .option("partitionType", "MONTH")
    .option("clusteredFields", "user_id")
    .mode("append")
    .save()
)

콘솔 → **BigQuery → spark_demo.orders_partitioned_spark → 세부정보** 탭에서 `Partitioned by` (`created_at`, 월) 와 `Clustered by` (`user_id`) 를 확인한다.

SQL 로도 확인:

```sql
SELECT column_name, is_partitioning_column, clustering_ordinal_position
FROM `YOUR_PROJECT_ID.spark_demo`.INFORMATION_SCHEMA.COLUMNS
WHERE table_name = 'orders_partitioned_spark'
  AND (is_partitioning_column = 'YES' OR clustering_ordinal_position IS NOT NULL);
```